# Training of GReaT (GPT-2) Model on the German Credit Risk Dataset

## Introduction

This notebook is a continuation of the workflow introduced in "Training of Generative Models on the German Credit Risk Dataset" (https://colab.research.google.com/drive/10ccgjsqsfisB8iWh4_2QwXShlvByPLO), focusing specifically on the training of the final generative model, GReaT with the full GPT-2 backbone.

The separation of this notebook from the others is primarily motivated by computational considerations: training transformer-based models, such as GPT-2, requires substantially more resources and time compared to traditional generative models. By isolating this process, we ensure that the workflow remains modular, manageable, and reproducible, allowing users to execute this resource-intensive step independently without affecting the other stages of the study.

## 0 Imports and function definition

This notebook uses a subset of essential libraries for data handling, model training, and persistence. pandas is employed for tabular data manipulation, GReaT from the be_great package provides transformer-based generative modeling, and pickle is used for saving and loading trained models. Their integration into the workflow will be demonstrated in the relevant sections of this notebook.

In [ ]:
!pip install be_great peft==0.9.0 sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.9/13.9 MB 111.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.5/193.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
import os
from be_great import GReaT
import pickle

In [ ]:
os.environ["WANDB_DISABLED"] = "true" #to avoid the requests from WANDB

The preprocess_dataset function standardizes the data types of categorical and numerical features and handles missing values in a consistent manner. Categorical features are cast to object and missing entries are filled with "Unknown", while numerical features are cast to float and missing values are replaced with the feature mean. This produces a clean dataset ready for training the GReaT model, following the same preprocessing approach used in the previous notebook.

In [ ]:
def preprocess_dataset(df, target_column, categorical_features, numerical_features):
    df = df.copy()

    for col in categorical_features:
        df[col] = df[col].astype('object')

    for col in numerical_features:
        df[col] = df[col].astype('float')

    for col in categorical_features:
        df[col] = df[col].fillna('Unknown')

    for col in numerical_features:
        df[col] = df[col].fillna(df[col].mean())

    return df


The train_and_save_model_GReaT function encapsulates the training of the GReaT model using the full GPT-2 backbone. Training parameters such as batch_size, epochs, and efficient_finetuning can be adjusted to control computational efficiency. Once training is complete, the model is serialized with pickle for subsequent use in synthetic data generation, following the modular and reproducible workflow established in the first notebook.

In [ ]:
def train_and_save_model_GReaT(df, batch_size, epochs, efficient_finetuning):

  filename = "GReaT_german.pkl"

  model = GReaT(llm="gpt2", batch_size=batch_size, epochs=epochs, fp16=True, efficient_finetuning=efficient_finetuning)
  model.fit(df)

  # Salvataggio modello
  with open(filename, 'wb') as f:
    pickle.dump(model, f, protocol=pickle.HIGHEST_PROTOCOL)

## 1 Dataset loading

The German Credit Risk dataset is loaded from its CSV file into a pandas DataFrame. The first column, serving as an index, is removed to simplify further processing. Based on the dataset documentation, categorical and numerical features are defined to facilitate preprocessing. The dataset is then passed through the preprocess_dataset function, which standardizes data types and imputes missing values, producing a clean dataset ready for training the GReaT model.

In [ ]:
german_df = pd.read_csv('german_credit_risk.csv')

In [ ]:
german_df = german_df.drop(german_df.columns[0], axis=1)

german_categorical_features = ['Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']
german_numerical_features = ['Age', 'Credit amount', 'Duration']

In [ ]:
preprocessed_dataset = preprocess_dataset(german_df, 'Risk', german_categorical_features, german_numerical_features)

/tmp/ipython-input-4-1313495981.py:11: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna('Unknown')


## 2 Train

Following the preprocessing step, we proceed to train the GReaT model using the custom function defined earlier. This encapsulated approach ensures that the model is trained on the cleaned dataset and subsequently serialized for later use, maintaining reproducibility and consistency within the workflow.

### GReaT

In [ ]:
train_and_save_model_GReaT(preprocessed_dataset, 32, 200, "lora")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/peft/utils/other.py:145: FutureWarning: prepare_model_for_int8_training is deprecated and will be removed in a future version. Use prepare_model_for_kbit_training instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/layer.py:861: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 589,824 || all params: 125,029,632 || trainable%: 0.4717473694555863


/usr/local/lib/python3.11/dist-packages/be_great/great.py:174: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `GReaTTrainer.__init__`. Use `processing_class` instead.
  great_trainer = GReaTTrainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,1.944400
1000,0.963800
1500,0.879300
2000,0.837800
2500,0.810600
3000,0.796500
3500,0.787300
4000,0.779500
4500,0.773800
5000,0.770900
